In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from autogluon.tabular import TabularPredictor

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)
#forward_fill_imputation(ts_data) 

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

full_dataset = NephroCAGEDataset(static_df=static_df, ts_data=ts_data, notes_df=notes, biopsy_df=dfs['biopsy'])
datapoints_limit = len(full_dataset)
#datapoints_limit = 320
dataset = Subset(full_dataset, indices=list(range(datapoints_limit)))
ts_scaler = full_dataset.ts_scaler
static_scaler = full_dataset.scaler

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


In [4]:
# Use ALL patients — encoder was trained self-supervised on all data (no label leakage)
# K-fold CV will be done on the classifier level with patient-level grouping
torch.manual_seed(42)
batch_size = 16
full_dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=full_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1

model.load_state_dict(torch.load('../models/full10ep.pt', weights_only=True))

<All keys matched successfully>

In [5]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=5
):
    """
    - Feeds each patient's entire time series (minus last step) in one pass.
    - Extracts hidden representations for each time step from model output.
    - For each horizon H, determines if the event occurs within H days from that step.
    - Only includes time steps between min_history_days and max_days.
    - Takes up to max_samples_per_patient samples per patient (first N valid samples).

    If `label_key` corresponds to a multi-day event list (like "rej_rel_days"), then `rel_days_key`
    can be left None (ignored), and we will handle the logic differently.
    
    Parameters:
    -----------
    dataloader : DataLoader
        The PyTorch dataloader yielding patient data batches
    model : torch.nn.Module
        The trained model to extract representations from
    horizons : list
        List of horizon values (in days) to consider
    label_key : str
        Key in batch dictionary for labels
    rel_days_key : str, optional
        Key for relative days to event (only for single-event labels)
    min_history_days : int, default=90
        Minimum number of days of history required
    max_days : int, default=180
        Maximum number of days to include in the dataset
    max_samples_per_patient : int, default=5
        Maximum number of samples to include per patient
    """
    from tqdm import tqdm
    
    model.eval()

    # For each horizon, prepare storage for hidden reps, labels, day-of-step, patient_id
    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}
    
    # Keep track of samples per patient for each horizon
    patient_sample_counts = {H: {} for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        # Add progress bar for the dataloader iteration
        for batch in tqdm(dataloader, desc="Processing patients"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            # Static features
            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            # Time series
            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)

            # --- Convert single-element Tensors in label_key to float/list ---
            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    # If it's a single element, convert to float
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        # If multi-element, convert to NumPy or list
                        val = val.cpu().numpy()
                # Otherwise, val can be float, int, list, etc.
                labels_data.append(val)

            # If single-event usage, we also have rel_days_key => shape (B,)
            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  # We'll handle multi-day logic below

            # If using notes
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                # label_or_list => single-event (float 0/1) or multi-event list
                label_or_list = labels_data[i]

                # Keep track of which patients have at least one event
                if isinstance(label_or_list, (int, float, np.number)):
                    # single label
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    # multiple days => if not empty => event
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    # Unknown type
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                # If the sequence is too short
                if slen[i] < 2:
                    continue
                
                # Check if we already have max samples for this patient for all horizons
                if max_samples_per_patient > 0:
                    all_horizons_at_max = True
                    for H in horizons:
                        if patient_id_i not in patient_sample_counts[H] or patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            all_horizons_at_max = False
                            break
                    
                    if all_horizons_at_max:
                        continue  # Skip this patient altogether if already at max for all horizons

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                elapsed_times = tm_i[:, 1:] - tm_i[:, :-1]
                inp_mask = mk_i[:, :-1]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i
                )
                # lstm_out => shape (1, seq_len_i-1, hidden_size)

                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Initialize counts for this patient if not present
                for H in horizons:
                    if patient_id_i not in patient_sample_counts[H]:
                        patient_sample_counts[H][patient_id_i] = 0
                
                # Loop through time steps and only process until we reach max_samples for each horizon
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k+1]  # day of the (k+1)-th step
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    # Get representation for this time step
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Check for each horizon if we still need more samples
                    any_horizon_needs_samples = False
                    for H in horizons:
                        if patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            any_horizon_needs_samples = True
                            break
                    
                    if not any_horizon_needs_samples:
                        break  # Exit time step loop if all horizons have enough samples
                    
                    # Process for each horizon that still needs samples
                    for H in horizons:
                        # Skip if already have max samples for this horizon
                        if patient_sample_counts[H][patient_id_i] >= max_samples_per_patient:
                            continue
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        # Else multi-event logic (like rejections)
                        else:
                            # label_or_list is a list of event days or None
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                # label_ = 1 if any day d in label_or_list satisfies (0 < d - cur_day <= H)
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        # Store
                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)
                        
                        # Update sample count
                        patient_sample_counts[H][patient_id_i] += 1

    # Convert lists to numpy arrays for convenience
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print statistics
    for H in horizons:
        total_patients = len(patient_sample_counts[H])
        avg_samples = np.mean([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]])
        max_samples = max([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]]) if patient_sample_counts[H] else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients: {len(all_pids)}, patients with event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [6]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720

print("Extracting representations for ALL patients...")

graft_repr, graft_lbl, graft_days, graft_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)

rej_repr, rej_lbl, rej_days, rej_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)

mort_repr, mort_lbl, mort_days, mort_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)

for H in horizons:
    print(f"\n===== Horizon {H} days =====")
    print(f"GraftLoss: {graft_repr[H].shape}, pos={int(graft_lbl[H].sum())}, rate={graft_lbl[H].mean():.4f}")
    print(f"Rejection: {rej_repr[H].shape}, pos={int(rej_lbl[H].sum())}, rate={rej_lbl[H].mean():.4f}")
    print(f"Mortality: {mort_repr[H].shape}, pos={int(mort_lbl[H].sum())}, rate={mort_lbl[H].mean():.4f}")

Extracting representations for ALL patients...


Processing patients: 100%|██████████| 212/212 [06:20<00:00,  1.79s/it]


Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 586


Processing patients: 100%|██████████| 212/212 [06:33<00:00,  1.86s/it]


Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 267


Processing patients: 100%|██████████| 212/212 [06:22<00:00,  1.80s/it]

Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 1038

===== Horizon 30 days =====
GraftLoss: (16198, 512), pos=35, rate=0.0022
Rejection: (16198, 512), pos=158, rate=0.0098
Mortality: (16198, 512), pos=73, rate=0.0045

===== Horizon 90 days =====
GraftLoss: (16198, 512), pos=113, rate=0.0070
Rejection: (16198, 512), pos=258, rate=0.0159
Mortality: (16198, 512), pos=170, rate=0.0105

===== Horizon 180 days =====
GraftLoss: (16198, 512), pos=183, rate=0.0113
Rejection: (16198, 512), pos=301, rate=0.0186
Mortality: (16198, 512), pos=292, rate=0.0180


In [7]:
# Logistic regression baseline (skipped — using GroupKFold MLP below)
pass

In [8]:
from sklearn.model_selection import StratifiedGroupKFold
import time


def train_mlp_fold(X_train, y_train, X_test, y_test,
                   event_name, epochs=30, batch_size=32, lr=5e-3, seed=42):
    """Train SimpleMLP on one fold. Returns best-epoch metrics + state dict."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_te = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)

    # Dynamic pos_weight from fold class ratio, capped at 50
    n_pos = max(float(y_train.sum()), 1.0)
    n_neg = len(y_train) - n_pos
    pw = min(n_neg / n_pos, 50.0)
    pos_weight = torch.tensor([pw]).to(device)

    mlp = SimpleMLP(input_dim=X_train.shape[1]).to(device)
    optimizer = optim.Adam(mlp.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        mlp.train()
        for bx, by in train_loader:
            optimizer.zero_grad()
            criterion(mlp(bx), by).backward()
            optimizer.step()

        # Evaluate every epoch for best-checkpoint tracking
        mlp.eval()
        with torch.no_grad():
            probs = torch.sigmoid(mlp(X_te)).cpu().numpy()
            y_true = y_te.cpu().numpy()

        unique = set(y_true.flatten())
        auc = roc_auc_score(y_true, probs) if len(unique) > 1 else float('nan')
        if not np.isnan(auc) and auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}

    # Compute full metrics at best checkpoint
    if best_state is not None:
        mlp.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    mlp.eval()
    with torch.no_grad():
        probs = torch.sigmoid(mlp(X_te)).cpu().numpy()
        y_pred = (probs >= 0.5).astype(int)
        y_true = y_te.cpu().numpy()

    auc = roc_auc_score(y_true, probs) if len(set(y_true.flatten())) > 1 else float('nan')
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return {
        'auc': auc, 'acc': acc, 'prec': prec, 'recall': rec,
        'spec': spec, 'f1': f1, 'pos_weight': pw,
        'n_train': len(y_train), 'n_test': len(y_test),
        'n_pos_train': int(y_train.sum()), 'n_pos_test': int(y_true.sum()),
        'best_state': best_state
    }


# ─── 5-Fold StratifiedGroupKFold CV ───
n_folds = 5
all_results = {}
t0 = time.time()

events = [
    ("GraftLoss", graft_repr, graft_lbl, graft_pids),
    ("Rejection", rej_repr, rej_lbl, rej_pids),
    ("Mortality", mort_repr, mort_lbl, mort_pids),
]

for H in horizons:
    print(f"\n{'='*60}")
    print(f"Horizon {H} days")
    print(f"{'='*60}")

    for evt_name, repr_dict, lbl_dict, pids_dict in events:
        full_name = f"{evt_name}@{H}"
        X_all = repr_dict[H]
        y_all = lbl_dict[H]
        pids = pids_dict[H]
        n_pos = int(y_all.sum())

        print(f"\n{full_name} (n={len(y_all)}, pos={n_pos}, rate={n_pos/len(y_all):.4f})")

        skf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=42)
        fold_results = []
        best_overall_auc = -1.0
        best_overall_state = None

        for fold_i, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all, groups=pids)):
            res = train_mlp_fold(
                X_all[tr_idx], y_all[tr_idx],
                X_all[te_idx], y_all[te_idx],
                event_name=full_name,
                epochs=30, batch_size=32, lr=5e-3, seed=42 + fold_i
            )
            fold_results.append(res)

            marker = ""
            if not np.isnan(res['auc']) and res['auc'] > best_overall_auc:
                best_overall_auc = res['auc']
                best_overall_state = res['best_state']
                marker = " *BEST*"

            print(f"  Fold {fold_i+1}/{n_folds}: AUC={res['auc']:.4f}, "
                  f"Prec={res['prec']:.4f}, Rec={res['recall']:.4f}, "
                  f"F1={res['f1']:.4f}, pw={res['pos_weight']:.1f}{marker}")

        aucs = [r['auc'] for r in fold_results if not np.isnan(r['auc'])]
        mean_auc = np.nanmean(aucs) if aucs else float('nan')
        std_auc = np.nanstd(aucs) if aucs else float('nan')
        print(f"  → {full_name}: AUC = {mean_auc:.4f} ± {std_auc:.4f}")

        if best_overall_state is not None:
            torch.save(best_overall_state, f'../models/{full_name}_clf.pth')
            print(f"  Saved best fold (AUC={best_overall_auc:.4f}) → ../models/{full_name}_clf.pth")

        all_results[full_name] = {"mean": mean_auc, "std": std_auc, "folds": fold_results}

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Total time: {elapsed/60:.1f} min")
print(f"\nSUMMARY: {n_folds}-Fold StratifiedGroupKFold CV (mean ± std AUC)")
print(f"{'='*60}")
print(f"{'Event':<20} {'30-day':<18} {'90-day':<18} {'180-day':<18}")
print("-" * 74)
for evt in ["GraftLoss", "Rejection", "Mortality"]:
    row = f"{evt:<20}"
    for H in horizons:
        key = f"{evt}@{H}"
        if key in all_results:
            row += f" {all_results[key]['mean']:.4f} ± {all_results[key]['std']:.4f}  "
        else:
            row += f" {'N/A':<16}"
    print(row)


Horizon 30 days

GraftLoss@30 (n=16198, pos=35, rate=0.0022)
  Fold 1/5: AUC=0.8608, Prec=0.0000, Rec=0.0000, F1=0.0000, pw=50.0 *BEST*
  Fold 2/5: AUC=0.6413, Prec=0.0000, Rec=0.0000, F1=0.0000, pw=50.0
  Fold 3/5: AUC=nan, Prec=0.0000, Rec=0.0000, F1=0.0000, pw=50.0
  Fold 4/5: AUC=0.9793, Prec=0.3000, Rec=0.2727, F1=0.2857, pw=50.0 *BEST*
  Fold 5/5: AUC=0.8847, Prec=0.0000, Rec=0.0000, F1=0.0000, pw=50.0
  → GraftLoss@30: AUC = 0.8415 ± 0.1238
  Saved best fold (AUC=0.9793) → ../models/GraftLoss@30_clf.pth

Rejection@30 (n=16198, pos=158, rate=0.0098)
  Fold 1/5: AUC=0.8534, Prec=0.0612, Rec=0.2143, F1=0.0952, pw=50.0 *BEST*
  Fold 2/5: AUC=0.8602, Prec=0.0357, Rec=0.0256, F1=0.0299, pw=50.0 *BEST*
  Fold 3/5: AUC=0.7927, Prec=0.0532, Rec=0.5526, F1=0.0970, pw=50.0
  Fold 4/5: AUC=0.8181, Prec=0.0568, Rec=0.4762, F1=0.1015, pw=50.0
  Fold 5/5: AUC=0.7768, Prec=0.0437, Rec=0.2188, F1=0.0729, pw=50.0
  → Rejection@30: AUC = 0.8202 ± 0.0327
  Saved best fold (AUC=0.8602) → ../models/

In [9]:
# Quick summary of k-fold results
print(f"SUMMARY: {n_folds}-Fold StratifiedGroupKFold CV (mean ± std AUC)")
print(f"{'='*74}")
print(f"{'Event':<20} {'30-day':<18} {'90-day':<18} {'180-day':<18}")
print("-" * 74)
for evt in ["GraftLoss", "Rejection", "Mortality"]:
    row = f"{evt:<20}"
    for H in horizons:
        key = f"{evt}@{H}"
        if key in all_results:
            row += f" {all_results[key]['mean']:.4f} ± {all_results[key]['std']:.4f}  "
        else:
            row += f" {'N/A':<16}"
    print(row)

SUMMARY: 5-Fold StratifiedGroupKFold CV (mean ± std AUC)
Event                30-day             90-day             180-day           
--------------------------------------------------------------------------
GraftLoss            0.8415 ± 0.1238   0.8705 ± 0.0807   0.8585 ± 0.0330  
Rejection            0.8202 ± 0.0327   0.7618 ± 0.0654   0.7973 ± 0.0292  
Mortality            0.8051 ± 0.0780   0.8535 ± 0.0576   0.8537 ± 0.0536  


In [10]:
clf_model = SimpleMLP(input_dim=512)
clf_model.load_state_dict(torch.load("../models/GraftLoss@90_clf.pth", weights_only=True))
clf_model.eval()
clf_model.to(device)

SimpleMLP(
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)

In [11]:
def train_and_eval_autogluon(X_train, y_train, X_test, y_test, event_name="Event", time_limit=60):
    """
    Train an AutoGluon model and evaluate its performance.
    
    Parameters:
    -----------
    X_train : numpy.ndarray
        Training features
    y_train : numpy.ndarray
        Training labels
    X_test : numpy.ndarray
        Test features
    y_test : numpy.ndarray
        Test labels
    event_name : str
        Name of the event being predicted
    time_limit : int
        Time limit in seconds for AutoGluon training
        
    Returns:
    --------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    metrics : dict
        Performance metrics
    """
    # Convert numpy arrays to pandas DataFrames
    feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]
    train_df = pd.DataFrame(X_train, columns=feature_names)
    test_df = pd.DataFrame(X_test, columns=feature_names)
    
    # Add labels
    train_df['label'] = y_train
    test_df['label'] = y_test
    
    # Check class distribution
    class_counts = np.bincount(y_train)
    print(f"Class distribution in training set: {class_counts}")
    
    # Calculate class weights for imbalance
    pos_scale = (len(y_train) / (2 * np.sum(y_train))) if np.sum(y_train) > 0 else 1.0
    neg_scale = (len(y_train) / (2 * (len(y_train) - np.sum(y_train)))) if len(y_train) - np.sum(y_train) > 0 else 1.0
    
    # Create directory for AutoGluon
    import os
    os.makedirs(f'models/{event_name}', exist_ok=True)
    
    # Initialize and train AutoGluon predictor
    print(f"Training AutoGluon model for {event_name}...")
    predictor = TabularPredictor(
        label='label',
        path=f'models/{event_name}',
        problem_type='binary',
        eval_metric='roc_auc', 
        verbosity=0,
    )
    
    # Train with hyperparameters focused on handling imbalanced data
    predictor.fit(
        train_data=train_df,
        time_limit=time_limit,  # time budget in seconds
        presets='good_quality',
        hyperparameters={
            'GBM': [
                {
                    'extra_trees': True,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                },
                {
                    'extra_trees': False,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                }
            ],
            'RF': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                },
                {
                    'criterion': 'entropy',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XT': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XGB': [
                {
                    'scale_pos_weight': pos_scale,  # Handle class imbalance
                    'max_depth': 6
                }
            ],
            'CAT': [
                {
                    'auto_class_weights': 'Balanced'  # Handle class imbalance
                }
            ],
            'FASTAI': [
                {
                    'weights': pos_scale,  # Handle class imbalance
                    'epochs': 20
                }
            ]
        },
        verbosity=0
    )
    
    # Make predictions on test data
    y_pred_proba = predictor.predict_proba(test_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Apply threshold of 0.5 for binary prediction
    threshold = 0.5
    y_pred = (y_pred_proba_pos >= threshold).astype(int)
    
    # Calculate metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # Print metrics
    print(f"\n{event_name} Prediction Results:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba_pos):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    
    # Save predictor leaderboard
    leaderboard = predictor.leaderboard(test_df, silent=True)
    print("\nModel Leaderboard:")
    print(leaderboard[['model', 'score_val', 'score_test']].head())
    
    # Return predictor and metrics
    metrics = {
        'auc': roc_auc_score(y_test, y_pred_proba_pos),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'specificity': specificity,
        'f1': f1_score(y_test, y_pred)
    }
    
    return predictor, metrics

def optimize_threshold(predictor, X_val, y_val):
    """
    Find optimal classification threshold based on validation data.
    
    Parameters:
    -----------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    X_val : numpy.ndarray
        Validation features
    y_val : numpy.ndarray
        Validation labels
        
    Returns:
    --------
    optimal_threshold : float
        Threshold that maximizes F1 score
    """
    # Convert to DataFrame
    feature_names = [f'feature_{i}' for i in range(X_val.shape[1])]
    val_df = pd.DataFrame(X_val, columns=feature_names)
    
    # Get predictions
    y_pred_proba = predictor.predict_proba(val_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Try different thresholds
    thresholds = np.linspace(0.1, 0.9, 9)
    f1_scores = []
    
    for threshold in thresholds:
        y_pred = (y_pred_proba_pos >= threshold).astype(int)
        f1 = f1_score(y_val, y_pred)
        f1_scores.append(f1)
    
    # Find threshold with best F1 score
    best_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[best_idx]
    
    print(f"Optimal threshold: {optimal_threshold:.2f} (F1: {f1_scores[best_idx]:.4f})")
    
    return optimal_threshold

In [12]:
import json

# Save k-fold CV results from cell 8 to JSON
final_results = {}
for key, val in all_results.items():
    final_results[key] = {
        "mean_auc": round(val["mean"], 4),
        "std_auc": round(val["std"], 4),
        "fold_aucs": [round(r["auc"], 4) for r in val["folds"]],
        "fold_metrics": [
            {
                "auc": round(r["auc"], 4),
                "acc": round(r["acc"], 4),
                "prec": round(r["prec"], 4),
                "recall": round(r["recall"], 4),
                "spec": round(r["spec"], 4),
                "f1": round(r["f1"], 4),
                "n_train": r["n_train"],
                "n_test": r["n_test"],
                "n_pos_test": r["n_pos_test"],
            }
            for r in val["folds"]
        ],
    }

with open("../data/results/final_res.json", "w") as f:
    json.dump(final_results, f, indent=2)

print("Results saved to data/results/final_res.json")
print("\n" + json.dumps(final_results, indent=2))

Results saved to data/results/final_res.json

{
  "GraftLoss@30": {
    "mean_auc": 0.8415,
    "std_auc": 0.1238,
    "fold_aucs": [
      0.8608,
      0.6413,
      NaN,
      0.9793,
      0.8847
    ],
    "fold_metrics": [
      {
        "auc": 0.8608,
        "acc": 0.9902,
        "prec": 0.0,
        "recall": 0.0,
        "spec": 0.9923,
        "f1": 0.0,
        "n_train": 12948,
        "n_test": 3250,
        "n_pos_test": 7
      },
      {
        "auc": 0.6413,
        "acc": 0.9932,
        "prec": 0.0,
        "recall": 0.0,
        "spec": 0.9944,
        "f1": 0.0,
        "n_train": 12985,
        "n_test": 3213,
        "n_pos_test": 4
      },
      {
        "auc": NaN,
        "acc": 0.9994,
        "prec": 0.0,
        "recall": 0.0,
        "spec": 0.9994,
        "f1": 0.0,
        "n_train": 12946,
        "n_test": 3252,
        "n_pos_test": 0
      },
      {
        "auc": 0.9793,
        "acc": 0.9954,
        "prec": 0.3,
        "recall": 0.2727,
 